# 8회차 실습 ① — MLflow로 여러 모델 비교하기 (기본)

`train_baseline.ipynb`를 참고해, **여러 회귀 모델을 학습하면서
그 성능을 MLflow에 기록하고 UI에서 비교**하는 실습입니다.

---

### 이 실습에서 하는 것

- **예측 타깃** : `daily_count` (일 이용대수)
- **비교 모델** : LinearRegression · Ridge · Lasso · RandomForest · GradientBoosting · XGBoost · LightGBM
- **기록 지표** : RMSLE(기준) · RMSE · MAE
- **핵심** : PyCaret 같은 AutoML 도구 없이, `for` 반복문으로 여러 모델을 돌려 MLflow에 남긴다

---

> 💡 **실행 전 준비**
>
> ```bash
> pip install mlflow
> ```
>
> 그 외 패키지(pandas, scikit-learn, xgboost, lightgbm)는 기존 학습 노트북과 동일합니다.

## 1. 라이브러리 임포트

데이터 로드(SQLAlchemy), 모델(scikit-learn / XGBoost / LightGBM),
그리고 실험 기록용 MLflow를 불러옵니다.

In [1]:
import os
from datetime import datetime   # 실행 날짜 태그용
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv          # .env 파일에서 DB 접속 정보 읽기
from sqlalchemy import create_engine, text   # MySQL 연결 + SQL 실행

# 비교할 회귀 모델들
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error  # 평가 지표 계산용
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 실험 추적 도구
import mlflow           # 실험(Run) 기록
import mlflow.sklearn   # scikit-learn 모델을 MLflow에 저장하는 헬퍼

# 설치된 MLflow 버전 확인(정상 임포트 확인용)
print("임포트 완료. MLflow 버전:", mlflow.__version__)

임포트 완료. MLflow 버전: 3.14.0


## 2. DB 연결 + 데이터 로드

기존 학습 노트북과 **동일한 방식**입니다.
`.env` 파일에 저장된 DB 접속 정보로 MySQL에 연결합니다.

In [2]:
load_dotenv()   # 현재 폴더(또는 상위)의 .env를 읽어 환경변수로 등록

# .env에 저장해 둔 DB 접속 정보를 하나씩 읽어온다
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

# SQLAlchemy 연결 문자열 조립: mysql+pymysql://아이디:비번@호스트:포트/DB이름
DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)   # DB 엔진 생성(실제 연결은 쿼리 시점에)

# 연결 확인: SELECT 1의 결과가 1이면 정상
with engine.connect() as conn:
    ok = conn.execute(text("SELECT 1")).scalar() == 1
    print("DB 연결 성공" if ok else "DB 연결 실패")

DB 연결 성공


### 학습 데이터 조회

`parking_daily`(일별 이용대수)에 주차장 정보·공휴일을 조인해
모델이 쓸 피처와 타깃을 한 번에 뽑습니다.

In [3]:
sql = """
    SELECT
        d.lot_id,                                   -- 주차장 ID
        i.capacity,                                 -- 총 주차면수
        MONTH(d.use_date)                       AS month,        -- 월(1~12): 계절성
        DAYOFWEEK(d.use_date)                   AS day_of_week,  -- 요일(1=일~7=토)
        CASE WHEN DAYOFWEEK(d.use_date) IN (1,7)
             THEN 1 ELSE 0 END                  AS is_weekend,   -- 주말 여부(일·토)
        WEEKOFYEAR(d.use_date)                  AS week_of_year, -- 연중 몇 번째 주
        COALESCE(h.holiday_date IS NOT NULL, 0) AS is_holiday,   -- 공휴일 여부
        d.use_date,                                 -- 날짜(정렬·분할용)
        d.daily_count                           AS daily_count   -- ★ 예측 타깃(일 이용대수)
    FROM parking_daily d
    INNER JOIN parking_info i ON d.lot_id   = i.id            -- 주차장 정보 조인
    LEFT  JOIN holidays     h ON d.use_date = h.holiday_date  -- 공휴일 조인(없으면 NULL)
    WHERE d.daily_count > 0 AND i.capacity > 0   -- 명백한 오류(0 이하) 제외
    ORDER BY d.use_date, d.lot_id                 -- 시간순 정렬(시계열 분할 위해)
"""

df = pd.read_sql(text(sql), engine)   # SQL 실행 결과를 DataFrame으로
print(f"로드 완료: {len(df):,}행")
df.head()   # 상위 5행 미리보기

로드 완료: 21,869행


,lot_id,capacity,month,day_of_week,is_weekend,week_of_year,is_holiday,use_date,daily_count
0,2,1782,1,3,0,3,0,2020-01-14,2
1,2,1782,1,4,0,3,0,2020-01-15,114
2,7,99,1,4,0,3,0,2020-01-15,12
3,2,1782,1,5,0,3,0,2020-01-16,873
4,7,99,1,5,0,3,0,2020-01-16,28


## 3. 피처 / 타깃 정의 + 시계열 분할

시계열 데이터는 무작위로 섞으면 **미래 정보가 학습에 새어드는** 문제가 생깁니다.
그래서 날짜 순서대로 앞 80%는 학습, 뒤 20%는 테스트로 나눕니다.

In [4]:
# 모델 입력 피처 7개 (순서 중요 — 예측 때도 이 순서를 지켜야 함)
FEATURE_COLUMNS = [
    "lot_id", "capacity", "month", "day_of_week",
    "is_weekend", "week_of_year", "is_holiday",
]
TARGET_COLUMN = "daily_count"   # 예측 대상: 일 이용대수

# 날짜 순으로 정렬(시계열 분할의 전제)
df_sorted = df.sort_values("use_date").reset_index(drop=True)
X = df_sorted[FEATURE_COLUMNS]   # 입력
y = df_sorted[TARGET_COLUMN]     # 정답

# 앞 80% = 학습, 뒤 20% = 테스트 (셔플하지 않음)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"훈련셋: {len(X_train):,}행 / 테스트셋: {len(X_test):,}행")

훈련셋: 17,495행 / 테스트셋: 4,374행


## 4. 평가 지표 함수

`train_baseline.ipynb`와 동일한 **RMSLE / RMSE / MAE** 를 씁니다.

- **RMSLE** : 로그 변환 후 오차 → 작은 값·큰 값의 상대 오차를 동등하게 평가(기준 지표)
- **RMSE** : 큰 오차에 민감
- **MAE** : "평균 몇 대 틀렸나"를 직관적으로 보여줌

In [5]:
def rmsle(y, pred):
    # 로그(log1p) 변환 후 RMSE. 값의 크기에 상관없이 '상대 오차'를 본다.
    log_y    = np.log1p(y)
    log_pred = np.log1p(np.maximum(pred, 0))   # 예측이 음수면 0으로 눌러 로그 오류 방지
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def rmse(y, pred):
    # 평균 제곱 오차의 제곱근
    return np.sqrt(mean_squared_error(y, pred))

def eval_metrics(y, pred):
    # 세 지표를 딕셔너리로 한 번에 반환(MLflow 기록에 그대로 사용)
    return {
        "rmsle": rmsle(y, pred),
        "rmse" : rmse(y, pred),
        "mae"  : mean_absolute_error(y, pred),
    }

## 5. MLflow 실험 설정

**Experiment**(실험 그룹)에 이름을 붙입니다.
이후 만들어지는 모든 Run이 이 그룹에 쌓이고, UI에서 이 이름으로 묶여 보입니다.

In [6]:
# 📌 방법 A — SQLite 백엔드 (MLflow 권장 방식)
#   파일 저장(file store) 대신 SQLite DB 파일에 실험을 기록한다.
#   - 현재 폴더에 mlflow.db 파일이 생기고 거기에 Run이 쌓인다.
#   - 파일 저장과 달리 최신 MLflow가 정식 지원하는 방식이다.
#   ⚠️ UI를 띄울 때 반드시 같은 백엔드를 지정해야 한다:
#       mlflow ui --backend-store-uri sqlite:///mlflow.db
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# 실험 그룹 지정(없으면 자동 생성)
mlflow.set_experiment("hangang_parking_model_compare_v2")
print("실험 설정 완료:", mlflow.get_tracking_uri())

실험 설정 완료: sqlite:///mlflow.db


## 6. 여러 모델을 반복문으로 학습하며 MLflow에 기록

이 실습의 **핵심**입니다. 모델마다 `mlflow.start_run()`으로 Run을 하나 만들고,
그 안에서 다음 세 가지를 남깁니다.

| 기록 | 함수 | 내용 |
| --- | --- | --- |
| 설정 | `log_param` | 어떤 모델·설정이었는지 |
| 성능 | `log_metric` | RMSLE / RMSE / MAE |
| 모델 | `log_model` | 학습된 모델 파일(나중에 UI에서 다운로드) |

이 반복문 하나가 곧 "여러 모델 자동 비교"입니다.

In [7]:
# 비교할 모델들: {표시이름: 추정기 객체}
models = {
    "LinearRegression": LinearRegression(),
    "Ridge"           : Ridge(alpha=1.0),
    "Lasso"           : Lasso(alpha=0.1),
    "RandomForest"    : RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost"         : XGBRegressor(n_estimators=300, random_state=42, verbosity=0),
    "LightGBM"        : LGBMRegressor(n_estimators=300, random_state=42, verbose=-1),
}

results = []   # 표로 정리하기 위해 결과를 모아둘 리스트

for name, model in models.items():
    try:
        # start_run: 이 블록이 하나의 실험(Run)이 된다. 블록을 벗어나면 자동 종료.
        with mlflow.start_run(run_name=name):
            model.fit(X_train, y_train)              # 1) 학습
            pred    = model.predict(X_test)          # 2) 테스트셋 예측
            metrics = eval_metrics(y_test, pred)     #    RMSLE/RMSE/MAE 계산

            # 3) MLflow에 기록 (모델 파일은 저장하지 않음 - 비교용이므로 지표/설정만)
            mlflow.log_param("model", name)          # 어떤 모델인지
            mlflow.log_param("target", TARGET_COLUMN)  # 무엇을 예측했는지
            # 모델의 모든 하이퍼파라미터를 한 번에 기록(UI Parameters에 표시됨)
            mlflow.log_params(model.get_params())
            # 실행 날짜 태그(UI에서 tags.run_date로 날짜별 필터 가능)
            mlflow.set_tag("run_date", datetime.now().strftime("%Y-%m-%d %H:%M"))
            mlflow.log_metric("rmsle", metrics["rmsle"])
            mlflow.log_metric("rmse",  metrics["rmse"])
            mlflow.log_metric("mae",   metrics["mae"])

            results.append({"name": name, **metrics})
            print(f"[{name}] RMSLE={metrics['rmsle']:.4f}  RMSE={metrics['rmse']:.1f}  MAE={metrics['mae']:.1f}")
    except Exception as e:
        # 한 모델이 실패해도 나머지는 계속 진행
        print(f"[{name}] 실패 -> 건너뜀: {e}")

print("\n모든 모델 기록 완료 -> MLflow UI에서 비교하세요.")

[LinearRegression] RMSLE=0.8708  RMSE=1253.4  MAE=839.7
[Ridge] RMSLE=0.8707  RMSE=1253.4  MAE=839.7
[Lasso] RMSLE=0.8702  RMSE=1253.4  MAE=839.7
[RandomForest] RMSLE=0.6691  RMSE=915.7  MAE=637.3
[GradientBoosting] RMSLE=0.6525  RMSE=877.7  MAE=610.2
[XGBoost] RMSLE=0.6959  RMSE=896.2  MAE=617.6
[LightGBM] RMSLE=0.6539  RMSE=877.9  MAE=604.2

모든 모델 기록 완료 -> MLflow UI에서 비교하세요.


## 7. 결과를 표로 정리

RMSLE가 낮을수록 좋은 모델입니다. 오름차순으로 정렬해 한눈에 봅니다.

In [8]:
# 결과 리스트를 DataFrame으로 만들고 RMSLE 오름차순 정렬
results_df = pd.DataFrame(results).sort_values("rmsle").reset_index(drop=True)
results_df

,name,rmsle,rmse,mae
0,GradientBoosting,0.652517,877.704803,610.161229
1,LightGBM,0.653885,877.897841,604.175782
2,RandomForest,0.669127,915.653713,637.345495
3,XGBoost,0.695925,896.154214,617.557983
4,Lasso,0.870241,1253.371618,839.656509
5,Ridge,0.870679,1253.408227,839.721459
6,LinearRegression,0.870842,1253.411699,839.739624


## 8. MLflow UI로 비교하기

터미널에서 **노트북과 같은 폴더**에 이동한 뒤 실행합니다.

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

> ⚠️ SQLite 백엔드는 UI를 띄울 때 **반드시 `--backend-store-uri`** 를 붙여야
> 노트북이 기록한 `mlflow.db`를 읽습니다. 그냥 `mlflow ui`만 실행하면
> 빈 화면(다른 저장소)이 보입니다.

그다음 브라우저에서 아래 순서로 비교합니다.

```
1. http://127.0.0.1:5000 접속
2. 왼쪽에서 'hangang_parking_model_compare' 실험 선택
3. 비교할 Run들을 체크박스로 선택
4. [Compare] 버튼 클릭 → RMSLE/RMSE/MAE 를 나란히 비교
5. 가장 좋은 Run → [Artifacts] → model 에서 모델 파일 확인·다운로드
```

---

### ✅ 정리

- PyCaret 같은 AutoML 없이, `for` 반복문 + `mlflow.start_run()` 만으로
  여러 모델을 학습·기록·비교했습니다.
- 지표는 이 프로젝트 기준인 **RMSLE / RMSE / MAE** 입니다(R²가 아님).
- 다음 노트북(②)에서는 **로그 타깃 + 앙상블**까지 MLflow로 비교합니다.

In [9]:
# 기존에 떠 있는 MLflow UI를 종료, 터미널에서
# lsof -ti:5000 | xargs kill -9